In [1]:
import pandas as pd
import networkx as nx
from sklearn.model_selection import GroupShuffleSplit

# 1. Load your dataset
df = pd.read_csv("../preprocessing/train_data_with_preprocessing2.csv")

# 2. Build the Bipartite Graph
G = nx.Graph()

# Add prefixes to avoid collisions between identical context and reply strings
df['ctx_node'] = "CTX_" + df['prev_clean'].astype(str)
df['rep_node'] = "REP_" + df['target_clean'].astype(str)

# Create edges based on rows
edges = list(zip(df['ctx_node'], df['rep_node']))
G.add_edges_from(edges)

# 3. Find Connected Components
components = list(nx.connected_components(G))

# Map every node to a unique Component ID
node_to_component = {}
for component_id, nodes in enumerate(components):
    for node in nodes:
        node_to_component[node] = component_id

# 4. Map the Component IDs back to the original dataframe
# (You only need to check the context node, as the reply node is guaranteed to have the same ID)
df['component_id'] = df['ctx_node'].map(node_to_component)

# 5. Split the data ensuring components are kept together
# GroupShuffleSplit ensures all rows with the same component_id stay in the same split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, val_idx = next(gss.split(df, groups=df['component_id']))

train_df = df.iloc[train_idx].drop(columns=['ctx_node', 'rep_node', 'component_id', 'split', 'group', 'is_toxic', 'label_id', 'previous_reply', 'target_reply'])
val_df = df.iloc[val_idx].drop(columns=['ctx_node', 'rep_node', 'component_id', 'split', 'group', 'is_toxic', 'label_id', 'previous_reply', 'target_reply'])

print(f"Training rows: {len(train_df)}")
print(f"Validation rows: {len(val_df)}")

Training rows: 17991
Validation rows: 1440


In [2]:
train_df.rename(columns={'prev_clean': 'context', 'target_clean': 'reply'}, inplace=True)
val_df.rename(columns={'prev_clean': 'context', 'target_clean': 'reply'}, inplace=True)

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
  accuracy_score,
  f1_score,
  classification_report,
  confusion_matrix
)

In [3]:
target_col = 'label'

In [4]:
# Extract text columns for TF-IDF
X_context_train = train_df['context'].fillna("").astype(str)
X_reply_train = train_df['reply'].fillna("").astype(str)
y_train = train_df[target_col].fillna("nan").astype(str)

X_context_dev = val_df['context'].fillna("").astype(str)
X_reply_dev = val_df['reply'].fillna("").astype(str)
y_dev = val_df[target_col].fillna("nan").astype(str)

In [5]:
# Word-level TF-IDF
context_word = TfidfVectorizer(
  analyzer="word",
  ngram_range=(1, 2),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

reply_word = TfidfVectorizer(
  analyzer="word",
  ngram_range=(1, 2),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

In [6]:
# Character-level TF-IDF
context_char = TfidfVectorizer(
  analyzer="char_wb",
  ngram_range=(3, 5),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

reply_char = TfidfVectorizer(
  analyzer="char_wb",
  ngram_range=(3, 5),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

In [7]:
# Fit only on training data, transform both
Cw_train = context_word.fit_transform(X_context_train)
Cw_dev = context_word.transform(X_context_dev)

Rw_train = reply_word.fit_transform(X_reply_train)
Rw_dev = reply_word.transform(X_reply_dev)

Cc_train = context_char.fit_transform(X_context_train)
Cc_dev = context_char.transform(X_context_dev)

Rc_train = reply_char.fit_transform(X_reply_train)
Rc_dev = reply_char.transform(X_reply_dev)

In [8]:
Cw_train.shape, Rw_train.shape, Cc_train.shape, Rc_train.shape

((17991, 5207), (17991, 7527), (17991, 9850), (17991, 13525))

In [9]:
# Combine all four sparse feature matrices
X_train = hstack([Cw_train, Cc_train, Rw_train, Rc_train], format="csr")
X_dev = hstack([Cw_dev, Cc_dev, Rw_dev, Rc_dev], format="csr")

print("Training feature matrix:", X_train.shape)
print("Development feature matrix:", X_dev.shape)

Training feature matrix: (17991, 36109)
Development feature matrix: (1440, 36109)


In [16]:
import optuna
import numpy as np
from sklearn.metrics import log_loss

def objective(trial):
  # 1. Sample solver from a CONSTANT list to fix Optuna's dynamic space error
#   solver = trial.suggest_categorical("solver", ["lbfgs", "newton-cg", "sag", "newton-cholesky"])

  # 2. Sample regularization style from a CONSTANT list
  reg_type = trial.suggest_categorical("reg_type", ["none", "l2"])  ## "lbfgs" only supports "none" and "l2"

  # 3. Handle solver/penalty incompatibilities cleanly
#   if reg_type in ["l1", "elasticnet"] and solver != "saga":
#     # Tell Optuna to safely abort and skip this invalid trial
#     raise optuna.exceptions.TrialPruned() 

  # 4. Map to Scikit-Learn's new API to fix the deprecation warning
  # 4. Map to Scikit-Learn's new API
  if reg_type == "none":
      C = np.inf
      l1_ratio = 0.0  # Changed from None to 0.0 to prevent TypeError
  else:
      C = trial.suggest_float("C", 1e-4, 1e2, log=True)
      if reg_type == "l2":
          l1_ratio = 0.0

  # 5. Other hyperparameters
  max_iter = trial.suggest_int("max_iter", 500, 5000, step=500)
  
  cw_choice = trial.suggest_categorical("class_weight", ["balanced", "none"])
  class_weight = "balanced" if cw_choice == "balanced" else None

  # 6. Initialize model WITHOUT the 'penalty' argument
  model = LogisticRegression(
      solver="lbfgs",
      C=C,
      l1_ratio=l1_ratio,
      max_iter=max_iter,
      class_weight=class_weight,
      random_state=42,
  )

  model.fit(X_train, y_train)
  # y_pred = model.predict(X_dev)
  # score = accuracy_score(y_dev, y_pred) 
  y_proba = model.predict_proba(X_dev)
  score = log_loss(y_dev, y_proba)  # Use log loss for evaluation

  return score

In [17]:
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))

[I 2026-11-01 15:58:23,071] A new study created in memory with name: no-name-9f79e171-d891-4005-81b3-983c9f39bf54


In [18]:
study.optimize(objective, n_trials=50)

[I 2026-11-01 15:58:26,662] Trial 0 finished with value: 1.5231814686958702 and parameters: {'reg_type': 'l2', 'C': 2.465832945854912, 'max_iter': 3000, 'class_weight': 'balanced'}. Best is trial 0 with value: 1.5231814686958702.
[I 2026-11-01 15:58:29,156] Trial 1 finished with value: 1.8493034878480792 and parameters: {'reg_type': 'l2', 'C': 0.4042872735027334, 'max_iter': 4000, 'class_weight': 'none'}. Best is trial 0 with value: 1.5231814686958702.
[I 2026-11-01 15:58:31,319] Trial 2 finished with value: 3.4381561974106374 and parameters: {'reg_type': 'none', 'max_iter': 1000, 'class_weight': 'none'}. Best is trial 0 with value: 1.5231814686958702.
[I 2026-11-01 15:58:33,264] Trial 3 finished with value: 4.604250220424178 and parameters: {'reg_type': 'none', 'max_iter': 1500, 'class_weight': 'balanced'}. Best is trial 0 with value: 1.5231814686958702.
[I 2026-11-01 15:58:35,269] Trial 4 finished with value: 2.300016816002109 and parameters: {'reg_type': 'l2', 'C': 0.054502936945582

In [19]:
study.best_params

{'reg_type': 'l2',
 'C': 0.3134369297229209,
 'max_iter': 2000,
 'class_weight': 'balanced'}

In [10]:
best_model = LogisticRegression(
  solver="lbfgs",
  C=0.313,
  l1_ratio=0.0,  # Since we are only using "none"
  max_iter=2000,
  class_weight="balanced",
  random_state=42,
)

best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_dev)

print("Accuracy", accuracy_score(y_dev, y_pred))
print("F1 Score", f1_score(y_dev, y_pred, average="weighted"))
print("Classification Report:\n", classification_report(y_dev, y_pred))

Accuracy 0.5041666666666667
F1 Score 0.4477220446662571
Classification Report:
                                      precision    recall  f1-score   support

                     Dehumanization       0.82      0.09      0.16       690
                         Harassment       0.50      1.00      0.67         2
                        Hate Speech       0.11      0.78      0.19        27
                    Identity Attack       1.00      0.98      0.99        95
 Incitement / Encouragement of Harm       0.00      0.00      0.00         2
                             Insult       0.27      0.85      0.41       130
                 Mockery / Ridicule       0.99      0.96      0.97        81
                          Profanity       0.56      0.36      0.44        25
Sexual Harassment / Objectification       0.52      0.30      0.38        40
                             Threat       0.96      0.90      0.93        50
      Toxic Endorsement / Agreement       1.00      1.00      1.00      

c:\Users\rajde\anaconda3\envs\pytrch310\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\rajde\anaconda3\envs\pytrch310\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\rajde\anaconda3\envs\pytrch310\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

In [11]:
df_test = pd.read_csv("../preprocessing/test_data_with_preprocessing2.csv")
y_test = df_test[target_col].fillna("nan").astype(str)
X_test_context = df_test['context_clean'].fillna("").astype(str)
X_test_reply = df_test['reply_clean'].fillna("").astype(str)

In [12]:
Cw_test = context_word.transform(X_test_context)
Rw_test = reply_word.transform(X_test_reply)

Cc_test = context_char.transform(X_test_context)
Rc_test = reply_char.transform(X_test_reply)

X_test = hstack([Cw_test, Cc_test, Rw_test, Rc_test], format="csr")

In [14]:
print("Test Accuracy:", accuracy_score(y_test, best_model.predict(X_test)))

Test Accuracy: 0.34811529933481156
